# 🔍 Loser Autopsy Notebook

Systematic forensic analysis of trading failures.

```
   Hypothesize          Engineer              Validate
┌────────────┐      ┌────────────────┐    ┌────────────┐
│ "Losers    │ ───▶ │ Add @feature   │ ─▶ │ Re-run     │
│  happen    │      │ in Section 3   │    │ all cells  │
│  when X"   │      │ nothing else   │    │ check AUC  │
└────────────┘      └────────────────┘    └─────┬──────┘
     ▲                                          │
     └──────────────── iterate ─────────────────┘
```

**Workflow:** add one `@feature` function → re-run from Section 4 onward.

## 1. Setup & Data Loading

In [ ]:
import sys, os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi']     = 100
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
training_df = pd.read_hdf('all_trades_with_features.h5', 'data')

def repair_trade_dates(s):
    s = pd.to_datetime(s)
    if len(s) and s.dt.year.max() < 1990:
        s = pd.to_datetime(s.astype('int64'), unit='us')
    return s.dt.normalize()

training_df['entry_date'] = repair_trade_dates(training_df['entry_date'])
training_df['exit_date']  = repair_trade_dates(training_df['exit_date'])

for c in ['symbol','entry_date','exit_date','net_return','Y']:
    assert c in training_df.columns, f'Missing: {c}'

print(f"Trades: {len(training_df)}  |  {training_df['entry_date'].min()} → {training_df['entry_date'].max()}")
print(f"Win rate: {training_df['Y'].mean():.2%}  |  Mean return: {training_df['net_return'].mean():.3%}")

In [ ]:
# Load price panel once — referenced as `stocks` by all @feature functions
REMOTE_HOST = 'http://192.168.1.30:9000'
stocks = None

def _load_from_delta():
    from deltalake import DeltaTable
    opts = {
        'AWS_ACCESS_KEY_ID': 'CzOwnLkEDXQy951AOqes',
        'AWS_SECRET_ACCESS_KEY': 'fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S',
        'AWS_ENDPOINT_URL': REMOTE_HOST,
        'AWS_ALLOW_HTTP': 'true', 'AWS_EC2_METADATA_DISABLED': 'true',
        'AWS_REGION': 'us-east-1', 'aws_conditional_put': 'etag',
    }
    dt  = DeltaTable('s3://delta-table-storage/stocks', storage_options=opts)
    wl  = pd.read_csv('../backend/models/watchlist.csv').iloc[:, 0].values
    start = pd.Timestamp.now() - pd.DateOffset(years=10)
    df = dt.to_pandas(
        filters=[('date','>=',start),('symbol','in',wl)],
        columns=['symbol','date','close','open','high','low','volume'],
    )
    df = df.drop_duplicates(subset=['date','symbol'], keep='last')
    return df.set_index(['date','symbol']).unstack(level=1).bfill().ffill()

local_file = 'stocks_data_latest.h5'
try:
    if os.path.exists(local_file):
        print(f'Loading from {local_file}')
        with pd.HDFStore(local_file, mode='r') as store:
            stocks = store['stocks']
    else:
        print('Fetching from Delta Lake...')
        stocks = _load_from_delta()
        with pd.HDFStore(local_file, mode='w') as store:
            store.put('stocks', stocks)
        print(f'Saved to {local_file}')
    print(f'Stocks panel: {stocks.shape}')
except Exception as e:
    print(f'WARNING: Could not load stocks panel: {e}')
    print('  Path features and VNINDEX features will be skipped.')

## 2. Loser Segmentation

Segment by return severity — catastrophic losers need separate treatment.

In [ ]:
def segment_trades(df):
    df = df.copy()
    q = df['net_return'].quantile([0.10, 0.30, 0.70, 0.90]).values
    def cat(r):
        if   r <= q[0]: return '1_catastrophic_loss'
        elif r <= q[1]: return '2_medium_loss'
        elif r <= q[2]: return '3_marginal'
        elif r <= q[3]: return '4_medium_win'
        else:           return '5_big_win'
    df['outcome'] = df['net_return'].apply(cat)
    return df, q

training_df, return_quantiles = segment_trades(training_df)
print(f"Thresholds  10%={return_quantiles[0]:.3%}  30%={return_quantiles[1]:.3%}  "
      f"70%={return_quantiles[2]:.3%}  90%={return_quantiles[3]:.3%}")
print(training_df.groupby('outcome').agg(
    count=('net_return','count'), mean=('net_return','mean'), std=('net_return','std'),
).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bins = np.linspace(training_df['net_return'].quantile(0.01),
                   training_df['net_return'].quantile(0.99), 80)
pal  = {'1_catastrophic_loss':'#8B0000','2_medium_loss':'#DC143C',
        '3_marginal':'#808080','4_medium_win':'#90EE90','5_big_win':'#006400'}
for outcome, color in pal.items():
    axes[0].hist(training_df[training_df['outcome']==outcome]['net_return'],
                 bins=bins, alpha=0.6, label=outcome, color=color)
axes[0].axvline(0, color='black', alpha=0.5)
axes[0].set(xlabel='Net Return', title='Distribution by Segment')
axes[0].legend(fontsize=8)
training_df.boxplot(column='net_return', by='outcome', ax=axes[1], patch_artist=False)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set(title='Box Plot by Segment', ylabel='Net Return')
plt.suptitle(''); plt.tight_layout(); plt.show()

## 3. Feature Registry & Engineering

**This is the only section you edit when testing a new hypothesis.**

| You do | Happens automatically |
|---|---|
| Add `@feature("name", "description")` + function | Appears in §4 discrimination, §10 clustering, §13 export |
| Re-run §3 apply cell + remaining cells | All analysis refreshes with the new column |

**Function contract:**
- Input: `df` — full `training_df`
- Output: `pd.Series` (one column, name = decorator `name`) **or** `pd.DataFrame` (multiple columns)
- Use only data available at `df['entry_date']` — no look-ahead
- `stocks` is a notebook global (loaded in §1)

In [ ]:
# Registry — populated by @feature decorator
FEATURES: dict = {}

def feature(name: str, description: str = ''):
    """Register a feature function.
    fn(df) -> pd.Series   : adds one column named `name`
    fn(df) -> pd.DataFrame: adds all columns in the returned DataFrame
    """
    def decorator(fn):
        FEATURES[name] = {'fn': fn, 'description': description or name}
        return fn
    return decorator

print('Feature registry ready.')

In [ ]:
# ── Built-in: path features (MFE, MAE, efficiency, holding_days) ─────────────

@feature('path_features', 'MFE, MAE, efficiency, holding_days per trade')
def _(df):
    if stocks is None:
        return pd.DataFrame(
            {'mfe':np.nan,'mae':np.nan,'efficiency':np.nan,'holding_days':np.nan},
            index=df.index)
    close = stocks['close'].copy()
    close.index = pd.DatetimeIndex(close.index).normalize()
    mfe_list, mae_list = [], []
    for _, trade in df.iterrows():
        sym = trade['symbol']
        t0  = pd.Timestamp(trade['entry_date']).normalize()
        t1  = pd.Timestamp(trade['exit_date']).normalize()
        if sym not in close.columns:
            mfe_list.append(np.nan); mae_list.append(np.nan); continue
        path = close[sym].loc[t0:t1].dropna()
        if path.empty or float(path.iloc[0]) <= 0:
            mfe_list.append(np.nan); mae_list.append(np.nan); continue
        ep = float(path.iloc[0])
        mfe_list.append(float(path.max() / ep - 1))
        mae_list.append(float(path.min() / ep - 1))
    mfe = pd.Series(mfe_list, index=df.index)
    mae = pd.Series(mae_list, index=df.index)
    return pd.DataFrame({
        'mfe':          mfe,
        'mae':          mae,
        'efficiency':   df['net_return'] / mfe.replace(0, np.nan),
        'holding_days': (df['exit_date'] - df['entry_date']).dt.days,
    })


# ── Built-in: VNINDEX regime features ─────────────────────────────────────────

@feature('vnindex_features', 'VNINDEX momentum, trend, drawdown, volatility at entry')
def _(df):
    COLS = ['vnindex_ret_5d','vnindex_ret_20d','vnindex_above_ema50',
            'vnindex_above_ema200','vnindex_drawdown','vnindex_vol_20d']
    if stocks is None or 'VNINDEX' not in stocks['close'].columns:
        return pd.DataFrame({c: np.nan for c in COLS}, index=df.index)
    vix     = stocks['close']['VNINDEX'].dropna()
    vix_r5  = vix.pct_change(5)
    vix_r20 = vix.pct_change(20)
    vix_e50 = vix.rolling(50).mean()
    vix_e2  = vix.rolling(200).mean()
    vix_dd  = vix / vix.rolling(252).max() - 1
    vix_vol = vix.pct_change().rolling(20).std()
    def _at(s, t):
        try:    return float(s.loc[:t].iloc[-1])
        except: return np.nan
    rows = []
    for t in df['entry_date']:
        t = pd.Timestamp(t).normalize()
        v, e50, e200 = _at(vix,t), _at(vix_e50,t), _at(vix_e2,t)
        rows.append({
            'vnindex_ret_5d':       _at(vix_r5, t),
            'vnindex_ret_20d':      _at(vix_r20, t),
            'vnindex_above_ema50':  int(v>e50)  if not any(np.isnan([v,e50]))  else np.nan,
            'vnindex_above_ema200': int(v>e200) if not any(np.isnan([v,e200])) else np.nan,
            'vnindex_drawdown':     _at(vix_dd,  t),
            'vnindex_vol_20d':      _at(vix_vol, t),
        })
    return pd.DataFrame(rows, index=df.index)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ✏️  ADD YOUR HYPOTHESIS HERE                                           ║
# ║                                                                          ║
# ║  1. Write a @feature function below                                     ║
# ║  2. Re-run this cell + the _apply_all cell below                        ║
# ║  3. Re-run §4 onward — your feature appears automatically               ║
# ║                                                                          ║
# ║  Available globals: df, stocks, stocks['close']['VNINDEX']              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ─── Hypothesis: buying near the 52w high fails more (overextended entries) ───
@feature('dist_from_52w_high', 'close/52w-max − 1 at entry  (0=at peak, negative=below)')
def _(df):
    if stocks is None:
        return pd.Series(np.nan, index=df.index)
    cp = stocks['close']
    result = []
    for _, row in df.iterrows():
        sym = row['symbol']
        t   = pd.Timestamp(row['entry_date']).normalize()
        if sym not in cp.columns:
            result.append(np.nan); continue
        past = cp[sym].loc[:t].dropna().tail(252)
        result.append(float(past.iloc[-1] / past.max() - 1) if len(past) else np.nan)
    return pd.Series(result, index=df.index)


# ─── Hypothesis: low-volume breakouts fail more ───────────────────────────────
@feature('breakout_vol_ratio', 'entry volume / 20d avg volume  (>1 = confirmed)')
def _(df):
    if 'volume' not in df.columns or 'volume_ma20' not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return df['volume'] / df['volume_ma20'].replace(0, np.nan)


# ─── Hypothesis: weak intraday close signals failure ─────────────────────────
@feature('close_in_range', '(close−low)/(high−low)  — 1=top of range, 0=bottom')
def _(df):
    if not {'close','high','low'}.issubset(df.columns):
        return pd.Series(np.nan, index=df.index)
    return (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)


print(f"Registered {len(FEATURES)} features: {list(FEATURES.keys())}")

In [ ]:
# ── Apply all registered features to training_df ──────────────────────────────

def _apply_all(df):
    df = df.copy()
    print(f"Applying {len(FEATURES)} feature group(s)...")
    for name, h in FEATURES.items():
        try:
            result = h['fn'](df)
            if isinstance(result, pd.DataFrame):
                new = [c for c in result.columns if c not in df.columns]
                if not new:
                    print(f"  ↳ {name}: already present"); continue
                for c in new:
                    df[c] = result[c].values
                print(f"  ✓ {name} → {new}")
            else:
                if name in df.columns:
                    print(f"  ↳ {name}: already present"); continue
                df[name] = pd.Series(result, index=df.index).values
                print(f"  ✓ {name}")
        except Exception as e:
            print(f"  ✗ {name}: {e}")
    print(f"\ntraining_df: {df.shape}")
    return df

training_df = _apply_all(training_df)

# Rebuild cohorts with all new columns attached
losers       = training_df[training_df['outcome'].isin(['1_catastrophic_loss','2_medium_loss'])].copy()
winners      = training_df[training_df['outcome'].isin(['4_medium_win','5_big_win'])].copy()
catastrophic = training_df[training_df['outcome']=='1_catastrophic_loss'].copy()
print(f"Cohorts: losers={len(losers)}, winners={len(winners)}, catastrophic={len(catastrophic)}")

## 4. Univariate Feature Analysis

For each feature: how different are loser vs winner distributions? High Cohen's d = good model candidate.

In [ ]:
_EXCLUDE = {
    'symbol','entry_date','exit_date','date','col','entry_idx','exit_idx',
    'return','net_return','Y','outcome','metadata','type',
    'Entry Price','Exit Price','size','PnL','status','index',
    'open','high','low','close','volume',
    'direction','parent_id','id',
    'year_month','dow','month','hold_bin',
    'vix_ret_bin','vix_dd_bin','vix_vol_bin',
}
feature_cols = [
    c for c in training_df.columns
    if c not in _EXCLUDE
    and pd.api.types.is_numeric_dtype(training_df[c])
    and training_df[c].nunique() > 5
]
_registered = {n for n in FEATURES if n in training_df.columns}
# Registered features first so they're prominent in rankings
feature_cols = sorted(feature_cols, key=lambda c: (0 if c in _registered else 1, c))
print(f"Analyzing {len(feature_cols)} features ({len([c for c in feature_cols if c in _registered])} registered)")

In [ ]:
def feature_discrimination(df_loss, df_win, features):
    rows = []
    for f in features:
        a, b = df_loss[f].dropna().values, df_win[f].dropna().values
        if len(a) < 10 or len(b) < 10: continue
        pooled = np.sqrt((np.var(a,ddof=1) + np.var(b,ddof=1)) / 2)
        cohen_d = (np.mean(b) - np.mean(a)) / (pooled + 1e-9)
        ks_s, ks_p = stats.ks_2samp(a, b)
        try:    _, mwu_p = stats.mannwhitneyu(a, b, alternative='two-sided')
        except: mwu_p = 1.0
        rows.append({'feature':f, 'loser_mean':np.mean(a), 'winner_mean':np.mean(b),
                     'cohen_d':cohen_d, 'abs_cohen_d':abs(cohen_d),
                     'ks_stat':ks_s, 'ks_pvalue':ks_p, 'mwu_pvalue':mwu_p})
    return pd.DataFrame(rows).sort_values('abs_cohen_d', ascending=False)

discrim_df = feature_discrimination(losers, winners, feature_cols)
print("Top 15 most discriminative features:")
print(discrim_df.head(15).to_string(index=False))

In [ ]:
top12 = discrim_df.head(12)['feature'].tolist()
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, feat in zip(axes.flat, top12):
    lv, wv = losers[feat].dropna(), winners[feat].dropna()
    p1  = min(lv.quantile(0.01), wv.quantile(0.01))
    p99 = max(lv.quantile(0.99), wv.quantile(0.99))
    ax.hist(lv.clip(p1,p99), bins=40, alpha=0.5,
            label=f'Loser  μ={lv.mean():.3f}', color='red',   density=True)
    ax.hist(wv.clip(p1,p99), bins=40, alpha=0.5,
            label=f'Winner μ={wv.mean():.3f}', color='green', density=True)
    d = discrim_df.set_index('feature').loc[feat,'cohen_d']
    ax.set_title(f'{feat}\n(d={d:.3f})', fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Top Discriminative Features: Loser vs Winner', fontsize=14, y=1.00)
plt.tight_layout(); plt.show()

In [ ]:
print("Cohen d:  <0.2=trivial  0.2-0.5=small  0.5-0.8=medium  >0.8=large")
strong = discrim_df[discrim_df['abs_cohen_d'] > 0.5]
medium = discrim_df[(discrim_df['abs_cohen_d']>0.2)&(discrim_df['abs_cohen_d']<=0.5)]
weak   = discrim_df[discrim_df['abs_cohen_d'] <= 0.2]
print(f"Strong={len(strong)}  Medium={len(medium)}  Weak={len(weak)}")
if len(strong):
    print(strong[['feature','loser_mean','winner_mean','cohen_d']].to_string(index=False))

## 5. Catastrophic Loss Deep Dive

In [ ]:
catastrophic_vs_others = feature_discrimination(
    catastrophic,
    training_df[training_df['outcome']!='1_catastrophic_loss'],
    feature_cols,
)
print("Top 15 features distinguishing CATASTROPHIC losers:")
print(catastrophic_vs_others.head(15).to_string(index=False))

In [ ]:
top_cat = catastrophic_vs_others.head(8)['feature'].tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.flat, top_cat):
    cv = catastrophic[feat].dropna()
    ov = training_df[training_df['outcome']!='1_catastrophic_loss'][feat].dropna()
    p1  = min(cv.quantile(0.01), ov.quantile(0.01))
    p99 = max(cv.quantile(0.99), ov.quantile(0.99))
    ax.hist(ov.clip(p1,p99), bins=40, alpha=0.4, label='All other', color='steelblue', density=True)
    ax.hist(cv.clip(p1,p99), bins=30, alpha=0.6, label='Catastrophic', color='darkred', density=True)
    d = catastrophic_vs_others.set_index('feature').loc[feat,'cohen_d']
    ax.set_title(f'{feat} (d={d:.3f})', fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('What makes a CATASTROPHIC loss different?', fontsize=14, y=1.00)
plt.tight_layout(); plt.show()

## 6. Time & Regime Analysis

In [ ]:
training_df['year_month'] = training_df['entry_date'].dt.to_period('M')
monthly = training_df.groupby('year_month').agg(
    n_trades=('Y','count'), win_rate=('Y','mean'), avg_return=('net_return','mean'),
).reset_index()
monthly['year_month'] = monthly['year_month'].dt.to_timestamp()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].bar(monthly['year_month'], monthly['n_trades'], width=20, color='steelblue', alpha=0.7)
axes[0].set(ylabel='# Trades', title='Monthly Trade Activity')
axes[1].plot(monthly['year_month'], monthly['win_rate'], marker='o', color='darkgreen', alpha=0.7)
axes[1].axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
axes[1].fill_between(monthly['year_month'], 0, monthly['win_rate'],
                     where=(monthly['win_rate']<0.4), color='red', alpha=0.2)
axes[1].set(ylabel='Win Rate', ylim=(0,1), title='Monthly Win Rate')
axes[2].plot(monthly['year_month'], monthly['avg_return'], marker='o', color='steelblue', alpha=0.7)
axes[2].axhline(0, color='black', alpha=0.3)
axes[2].fill_between(monthly['year_month'],0,monthly['avg_return'],
                     where=(monthly['avg_return']<0),color='red',alpha=0.2)
axes[2].fill_between(monthly['year_month'],0,monthly['avg_return'],
                     where=(monthly['avg_return']>0),color='green',alpha=0.2)
axes[2].set(ylabel='Mean Return', xlabel='Date', title='Monthly Avg Return')
plt.tight_layout(); plt.show()

worst_months = monthly.nsmallest(10,'avg_return')
print("Top 10 worst months:")
print(worst_months.to_string(index=False))

In [ ]:
from scipy.stats import chi2_contingency
training_df['dow']   = training_df['entry_date'].dt.dayofweek
training_df['month'] = training_df['entry_date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dow_s = training_df.groupby('dow').agg(win_rate=('Y','mean')).reset_index()
axes[0].bar(dow_s['dow'][:5], dow_s['win_rate'][:5],
            color=['red' if w<training_df['Y'].mean() else 'green' for w in dow_s['win_rate'][:5]])
axes[0].axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
axes[0].set(xticks=range(5), xticklabels=['Mon','Tue','Wed','Thu','Fri'],
            ylabel='Win Rate', title='Win Rate by Day-of-Week')
mon_s = training_df.groupby('month').agg(win_rate=('Y','mean')).reset_index()
axes[1].bar(mon_s['month'], mon_s['win_rate'],
            color=['red' if w<training_df['Y'].mean() else 'green' for w in mon_s['win_rate']])
axes[1].axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
axes[1].set(ylabel='Win Rate', title='Win Rate by Month')
plt.tight_layout(); plt.show()

ct = pd.crosstab(training_df['dow'], training_df['Y'])
_, p, _, _ = chi2_contingency(ct)
print(f"Day-of-week chi2 p={p:.4f} — {'significant' if p<0.05 else 'not significant'}")

## 7. Symbol-Level Hotspots

In [ ]:
sym_stats = training_df.groupby('symbol').agg(
    n_trades=('Y','count'), win_rate=('Y','mean'),
    avg_return=('net_return','mean'), sum_return=('net_return','sum'),
).reset_index()
sym_stats = sym_stats[sym_stats['n_trades']>=10].sort_values('avg_return')
print("Worst 10:"); print(sym_stats.head(10).to_string(index=False))
print("\nBest 10:");  print(sym_stats.tail(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
combined = pd.concat([sym_stats.head(10), sym_stats.tail(10)])
axes[0].barh(combined['symbol'], combined['avg_return'],
             color=['darkred']*10+['darkgreen']*10)
axes[0].axvline(0, color='black')
axes[0].set(xlabel='Avg Net Return', title='Symbol Performance: Best vs Worst')
axes[1].hist(sym_stats['win_rate'], bins=30, alpha=0.7, edgecolor='black')
axes[1].axvline(sym_stats['win_rate'].mean(), color='black', linestyle='--',
                label=f"Mean: {sym_stats['win_rate'].mean():.2%}")
axes[1].axvline(0.5, color='red', linestyle=':')
axes[1].set(xlabel='Win Rate', ylabel='# Symbols', title='Per-Symbol Win Rate Distribution')
axes[1].legend()
plt.tight_layout(); plt.show()

chronic_losers = sym_stats[(sym_stats['win_rate']<0.35)&(sym_stats['n_trades']>=20)]
print(f"Chronic losers (winrate<35%, >=20 trades): {len(chronic_losers)}")
if len(chronic_losers):
    print(chronic_losers[['symbol','n_trades','win_rate','avg_return']].to_string(index=False))

## 8. Path Analysis: MFE & MAE

Data computed in §3 — visualization only.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0,0]
for grp, col, lbl in [(losers,'red','Losers'),(winners,'green','Winners')]:
    v = grp['mfe'].dropna()
    ax.hist(v.clip(-0.05,0.20), bins=40, alpha=0.5,
            label=f'{lbl} mu={v.mean():.2%}', color=col, density=True)
ax.axvline(0, color='black', alpha=0.3)
ax.set(xlabel='MFE', title='Did losers ever go positive?'); ax.legend()

ax = axes[0,1]
for grp, col, lbl in [(losers,'red','Losers'),(winners,'green','Winners')]:
    v = grp['mae'].dropna()
    ax.hist(v.clip(-0.20,0.05), bins=40, alpha=0.5,
            label=f'{lbl} mu={v.mean():.2%}', color=col, density=True)
ax.axvline(0, color='black', alpha=0.3)
ax.set(xlabel='MAE', title='How much did winners drawdown first?'); ax.legend()

ax = axes[1,0]
sample = training_df.dropna(subset=['mfe','mae']).sample(min(2000,len(training_df)))
sc = ax.scatter(sample['mae'], sample['mfe'], c=sample['Y'], cmap='RdYlGn', alpha=0.4, s=10)
ax.axhline(0, color='black', alpha=0.3); ax.axvline(0, color='black', alpha=0.3)
ax.set(xlabel='MAE', ylabel='MFE', title='Risk/Reward per Trade')
plt.colorbar(sc, ax=ax)

ax = axes[1,1]
hold_bins   = [0,5,10,20,40,80,200]
hold_labels = ['1-5d','6-10d','11-20d','21-40d','41-80d','80d+']
training_df['hold_bin'] = pd.cut(training_df['holding_days'], bins=hold_bins, labels=hold_labels)
hld = training_df.groupby('hold_bin', observed=True).agg(win_rate=('Y','mean'), n=('Y','count'))
ax.bar(range(len(hld)), hld['win_rate'],
       color=['red' if w<training_df['Y'].mean() else 'green' for w in hld['win_rate']])
ax.axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
ax.set(xticks=range(len(hld)), xticklabels=hld.index, ylabel='Win Rate', title='Win Rate by Holding Period')
for i,n in enumerate(hld['n']): ax.text(i, 0.05, f'n={n}', ha='center', color='white', fontweight='bold')

plt.tight_layout(); plt.show()
print(f"Losers ever profitable: {(losers['mfe']>0).mean():.1%}")
print(f"Median efficiency: {training_df['efficiency'].median():.2f}")

## 9. Visualize the Worst 9 Losers

In [ ]:
if stocks is None:
    print("WARNING: stocks panel not available")
else:
    worst9 = training_df.nsmallest(9,'net_return').reset_index(drop=True)
    fig, axes = plt.subplots(3, 3, figsize=(18, 12))
    for ax, (_,trade) in zip(axes.flat, worst9.iterrows()):
        sym,t0,t1 = trade['symbol'], trade['entry_date'], trade['exit_date']
        if sym not in stocks['close'].columns:
            ax.text(0.5,0.5,f'{sym}\nno data',ha='center',va='center'); continue
        idx = stocks.index
        cs  = idx[max(0, idx.get_loc(t0)-30)]
        ce  = idx[min(len(idx)-1, idx.get_loc(t1)+10)]
        full = stocks['close'][sym].loc[cs:ce]
        ax.plot(full.index, full.values, color='steelblue', lw=1)
        tp = stocks['close'][sym].loc[t0:t1]
        ax.plot(tp.index, tp.values, color='red', lw=2.5)
        ax.axvline(t0, color='green', linestyle='--', alpha=0.6)
        ax.axvline(t1, color='red',   linestyle='--', alpha=0.6)
        ax.set_title(f"{sym}: {trade['net_return']:.1%} / {(t1-t0).days}d", fontsize=10)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
    plt.suptitle('9 Worst Trades', fontsize=14, y=1.00)
    plt.tight_layout(); plt.show()

## 10. Pattern Mining: Clustering Losers

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

N_CLUSTERS  = 4
top_n       = 15
clust_feats = discrim_df.head(top_n)['feature'].tolist()

loser_data = losers[clust_feats].dropna()
scaler     = StandardScaler()
X_sc       = scaler.fit_transform(loser_data)
pca        = PCA(n_components=2)
X_pca      = pca.fit_transform(X_sc)
km         = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
clusters   = km.fit_predict(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for k in range(N_CLUSTERS):
    m = clusters==k
    axes[0].scatter(X_pca[m,0], X_pca[m,1], alpha=0.5, s=15, label=f'C{k}: {m.sum()}')
axes[0].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', title='Loser Clusters (PCA)')
axes[0].legend()
avg_ret = [losers.iloc[loser_data.index.get_indexer(loser_data.index)].iloc[clusters==k]['net_return'].mean()
           for k in range(N_CLUSTERS)]
axes[1].bar(range(N_CLUSTERS), avg_ret, color=['#FF6B6B','#4ECDC4','#FFE66D','#A8E6CF'][:N_CLUSTERS])
axes[1].set(xlabel='Cluster', ylabel='Avg Return', title='Severity per Cluster')
plt.tight_layout(); plt.show()

In [ ]:
centroids = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=clust_feats, index=[f'Cluster {k}' for k in range(N_CLUSTERS)],
)
centroids.loc['Winner mean'] = winners[clust_feats].mean()
print("Cluster centroids vs winner mean:")
print(centroids.round(3).to_string())

overall_std = training_df[clust_feats].std()
print("\nTop distinguishing features per cluster:")
for k in range(N_CLUSTERS):
    diff = (centroids.loc[f'Cluster {k}'] - centroids.loc['Winner mean']) / overall_std
    top5 = diff.abs().sort_values(ascending=False).head(5)
    print(f"\nCluster {k} (n={(clusters==k).sum()}):")
    for feat in top5.index:
        print(f"  {'up' if diff[feat]>0 else 'dn'} {feat}: {diff[feat]:+.2f}sigma")

## 11. VNINDEX Regime Diagnostics

VNINDEX features were pre-computed in §3 — visualization only.

In [ ]:
if 'vnindex_above_ema200' not in training_df.columns:
    print("WARNING: VNINDEX features not available")
else:
    print("Win rates by regime:")
    print(training_df.groupby(['vnindex_above_ema50','vnindex_above_ema200']).agg(
        n=('Y','count'), win_rate=('Y','mean'), avg_return=('net_return','mean')
    ).round(4))

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0,0]
    rb = pd.qcut(training_df['vnindex_ret_20d'].dropna(), 5,
                  labels=['Bot20%','20-40%','40-60%','60-80%','Top20%'])
    training_df['vix_ret_bin'] = rb
    bs = training_df.groupby('vix_ret_bin',observed=True).agg(win_rate=('Y','mean'))
    ax.bar(range(5), bs['win_rate'], color=['darkred','red','gray','green','darkgreen'])
    ax.axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
    ax.set(xticks=range(5), xticklabels=bs.index, ylabel='Win Rate', title='Win Rate by VNINDEX 20d Return')

    ax = axes[0,1]
    db = pd.qcut(training_df['vnindex_drawdown'].dropna(), 5)
    training_df['vix_dd_bin'] = db
    ds = training_df.groupby('vix_dd_bin',observed=True).agg(win_rate=('Y','mean'))
    ax.bar(range(5), ds['win_rate'], color=['darkred','red','gray','green','darkgreen'])
    ax.axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
    ax.set(xticks=range(5), xticklabels=[f'{b.left:.1%}' for b in ds.index],
           ylabel='Win Rate', title='Win Rate by VNINDEX Drawdown')

    ax = axes[1,0]
    vb = pd.qcut(training_df['vnindex_vol_20d'].dropna(), 5)
    training_df['vix_vol_bin'] = vb
    vs = training_df.groupby('vix_vol_bin',observed=True).agg(win_rate=('Y','mean'))
    ax.bar(range(5), vs['win_rate'], color=['darkgreen','green','gray','red','darkred'])
    ax.axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
    ax.set(xticks=range(5), xticklabels=[f'{b.left:.3f}' for b in vs.index],
           ylabel='Win Rate', title='Win Rate by VNINDEX Volatility')

    ax = axes[1,1]
    fs = training_df.groupby('vnindex_above_ema200').agg(
        n=('Y','count'), win_rate=('Y','mean'), avg_return=('net_return','mean'))
    fs.index = ['Bear (below EMA200)','Bull (above EMA200)']
    ax.bar(range(2), fs['win_rate'], color=['red','green'])
    ax.axhline(training_df['Y'].mean(), color='black', linestyle='--', alpha=0.5)
    ax.set(xticks=range(2), xticklabels=fs.index, ylabel='Win Rate', title='EMA200 Filter')
    for i,(_,row) in enumerate(fs.iterrows()):
        ax.text(i,0.05,f"n={row['n']}\n{row['avg_return']:.2%}",
                ha='center',color='white',fontweight='bold',fontsize=10)

    plt.tight_layout(); plt.show()

## 12. Feature Engineering Suggestions

In [ ]:
suggestions = []

if 'vnindex_above_ema200' in training_df.columns:
    bwr = training_df[training_df['vnindex_above_ema200']==0]['Y'].mean()
    bul = training_df[training_df['vnindex_above_ema200']==1]['Y'].mean()
    if abs(bul-bwr) > 0.05:
        suggestions.append({'priority':1,'name':'vnindex_trend_regime',
            'reason':f'Bull WR {bul:.1%} vs Bear WR {bwr:.1%}'})

if 'vix_vol_bin' in training_df.columns:
    vwr = training_df.groupby('vix_vol_bin',observed=True)['Y'].mean()
    if vwr.max()-vwr.min() > 0.05:
        suggestions.append({'priority':1,'name':'vnindex_volatility_regime',
            'reason':f'WR spans {vwr.min():.1%}-{vwr.max():.1%} across vol regimes'})

if len(chronic_losers):
    suggestions.append({'priority':1,'name':'symbol_blacklist / rolling_symbol_winrate',
        'reason':f'{len(chronic_losers)} symbols with <35% historical WR'})

if p < 0.05:
    suggestions.append({'priority':2,'name':'day_of_week','reason':'DOW chi2 significant'})

suggestions += [
    {'priority':1,'name':'breakout_quality',
     'reason':'dist_from_52w_high & close_in_range registered — check their Cohen d in section 4'},
    {'priority':2,'name':'cross_sectional_ranks',
     'reason':'RSI/vol/momentum ranks are orthogonal to price-level features'},
    {'priority':1,'name':'strategy_self_assessment',
     'reason':'Rolling WR of last N closed trades predicts near-term regime'},
]
suggestions.sort(key=lambda x: x['priority'])
print("=" * 60, "\nRANKED FEATURE SUGGESTIONS\n", "=" * 60)
for i,s in enumerate(suggestions,1):
    pri = 'HIGH' if s['priority']==1 else 'MED'
    print(f"\n{i}. [{pri}] {s['name']}\n   {s['reason']}")

## 13. Export Findings

In [ ]:
os.makedirs('../reports', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')
report = {
    'autopsy_date':       datetime.now().isoformat(),
    'registered_features': list(FEATURES.keys()),
    'data_summary': {
        'total_trades':       int(len(training_df)),
        'date_range':         [str(training_df['entry_date'].min()), str(training_df['entry_date'].max())],
        'symbols':            int(training_df['symbol'].nunique()),
        'overall_win_rate':   float(training_df['Y'].mean()),
        'overall_avg_return': float(training_df['net_return'].mean()),
    },
    'top_discriminative_features': discrim_df.head(20).to_dict('records'),
    'catastrophic_loss_drivers':   catastrophic_vs_others.head(20).to_dict('records'),
    'chronic_loser_symbols':       chronic_losers['symbol'].tolist() if len(chronic_losers) else [],
    'worst_months':                worst_months[['year_month','win_rate','avg_return']].astype(str).to_dict('records'),
    'feature_suggestions':         suggestions,
}
with open(f'../reports/loser_autopsy_{ts}.json','w') as f:
    json.dump(report, f, indent=2, default=str)
discrim_df.to_csv(f'../reports/discriminative_features_{ts}.csv', index=False)
print(f"  ../reports/loser_autopsy_{ts}.json")
print(f"  ../reports/discriminative_features_{ts}.csv")

## Next Steps

### Adding a hypothesis
1. Open §3 → "ADD YOUR HYPOTHESIS HERE" cell
2. Add a `@feature("name", "description")` function
3. Re-run §3 apply cell (`_apply_all`) + all cells below
4. Check §4 discrimination plot — does your feature rank in the top 12?
5. Cohen's d > 0.2 → add to meta-labeling feature pipeline

### Validation checklist
- [ ] No look-ahead (only data <= `entry_date`)
- [ ] Cohen's d > 0.2 in this autopsy
- [ ] OOS AUC improves >= 0.005 in meta-labeling notebook
- [ ] Sharpe lift positive in threshold sweep

### When to stop
Re-run each round. When no unmodeled feature has |d| > 0.3, shift focus to risk rules (position sizing, stops) rather than prediction.